[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/experiments/building_your_own_eval_harness.ipynb)


<center>
    <p style="text-align:center">
        <img alt="phoenix logo" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://docs.arize.com/phoenix/">Docs</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-1px8dcmlf-fmThhDFD_V_48oU7ALan4Q">Community</a>
    </p>
</center>


# Why public benchmarks lie: building your own eval harness

*Worked example: an email text-extraction service.*

## A leaderboard winner can lose on your task

When a new model tops MMLU, GPQA, or the latest leaderboard, it's tempting to assume it's the right choice for *your* application. It usually isn't — at least not for the reason you think. Public benchmarks measure **generic capabilities** on **generic data** with a **generic metric**: broad academic knowledge, scored as multiple-choice accuracy, averaged over thousands of questions that look nothing like your production traffic.

Your task is narrow and specific. You aren't asking the model trivia — you're asking it to pull five exact fields out of a customer email, every time, in a schema your downstream code can parse. A model that wins MMLU by two points can lose *your* task by twenty, because:

- **The data is different.** Your inputs are your customers' emails, not exam questions.
- **The metric is different.** You care whether the `due_date` field is exactly right, not whether the prose sounds smart.
- **The failure modes are different.** A confident, plausible-sounding wrong answer is worse for you than an "I don't know" — the opposite of what a knowledge benchmark rewards.

The only benchmark that predicts how a model performs on your task is **a benchmark built from your task.**

## What makes a benchmark trustworthy

Before trusting *any* number — public or your own — ask whether the benchmark behind it has these four properties:

1. **Representative data.** The examples are drawn from your real inputs, covering the cases you actually see (including the messy and ambiguous ones), not a clean toy sample.
2. **A metric that measures what you care about.** The score moves when the output gets better *for your purpose* and stays flat when it doesn't. The wrong metric can rank a worse model first — we'll show exactly this below.
3. **Enough examples to be stable.** Two examples can't distinguish two models; the score has to be more signal than noise.
4. **Reproducible and fair.** Every model is judged on the *same* dataset, with the *same* metric, under the *same* prompt — so a difference in the score reflects a difference in the model, not the setup.

A Phoenix **experiment** is exactly this harness: a fixed **dataset**, a **task** you vary, and one or more **evaluators**. Hold the dataset and evaluators constant, swap only the model, and the comparison is fair by construction.

**In this notebook, you will:**

- Build a small **domain dataset** of emails + their correct extractions — your benchmark, not a public one
- Define an **extraction task** with a fixed schema and prompt, parameterized only by model
- Define **two evaluators** — string similarity *and* field-level accuracy — and see how they can rank the models differently
- Run the **same harness** across `gpt-4o` and `gpt-4o-mini` and compare them fairly in Phoenix

✅ You will need a free [Phoenix Cloud account](https://app.arize.com/auth/phoenix/login) and an OpenAI API key to run this notebook.

# Set Up Keys & Dependencies

In [ ]:
%pip install arize-phoenix arize-phoenix-otel arize-phoenix-client openai openinference-instrumentation-openai jarowinkler nest_asyncio

In [ ]:
import os
from getpass import getpass

if not (phoenix_endpoint := os.getenv("PHOENIX_COLLECTOR_ENDPOINT")):
    phoenix_endpoint = getpass("🔑 Enter your Phoenix Collector Endpoint: ")
os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = phoenix_endpoint

if not (phoenix_api_key := os.getenv("PHOENIX_API_KEY")):
    phoenix_api_key = getpass("🔑 Enter your Phoenix API key: ")
os.environ["PHOENIX_API_KEY"] = phoenix_api_key

if not (openai_api_key := os.getenv("OPENAI_API_KEY")):
    openai_api_key = getpass("🔑 Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = openai_api_key

# Configure Tracing

Register a tracer provider and auto-instrument OpenAI so every extraction call shows up as a span in Phoenix. With tracing on, the experiment results link back to the exact calls that produced them.

In [ ]:
import nest_asyncio
from phoenix.otel import register

nest_asyncio.apply()  # allow running async experiment code inside the notebook

# auto_instrument=True activates the installed OpenInference instrumentors (here, OpenAI),
# so there's no need to call OpenAIInstrumentor().instrument(...) separately.
register(project_name="email-extraction-eval-harness", auto_instrument=True)

In [ ]:
from phoenix.client import AsyncClient

px_client = AsyncClient()

# Build a domain dataset

This is the part public benchmarks can't do for you. We hand-label a handful of emails the way our service actually sees them — a meeting request, an invoice, a support escalation, a sales reply — each paired with the **exact** structured output we want back.

In production you'd build this from real traffic (export traces from Phoenix, sample, and label). Here we inline a small set so the notebook is self-contained. Each example has an `email` input and an `expected` extraction with five fields:

- `sender` — who sent it (free text)
- `category` — one of a fixed set (categorical)
- `summary` — a one-line gist (free text)
- `action_required` — does this need a human to act? (boolean)
- `due_date` — an ISO date, or `"none"` (categorical-ish)

The mix of free-text and categorical fields matters: it's what lets two reasonable metrics *disagree* later.

> **This is a demonstration harness.** Eight examples is enough to illustrate the workflow, not to draw stable conclusions — recall property 3 above. A production harness needs a larger, representative sample drawn from your real traffic before you'd trust the ranking.

In [ ]:
EMAILS = [
    {
        "email": "From: Dana Whitfield <dana@acme.co>\nSubject: Kickoff for Q3 redesign\n\nHi team, can we lock in the kickoff for the Q3 site redesign? I'm proposing Thursday July 10 at 2pm PT. Please confirm by end of week so I can send invites.",
        "expected": {
            "sender": "Dana Whitfield",
            "category": "meeting",
            "summary": "Proposes a Q3 redesign kickoff meeting on July 10 at 2pm PT.",
            "action_required": True,
            "due_date": "2025-07-10",
        },
    },
    {
        "email": "From: billing@cloudhost.com\nSubject: Invoice #88231 due\n\nYour CloudHost invoice #88231 for $4,200.00 is due on 2025-06-30. Please remit payment to avoid a service interruption.",
        "expected": {
            "sender": "billing@cloudhost.com",
            "category": "invoice",
            "summary": "CloudHost invoice #88231 for $4,200 is due June 30.",
            "action_required": True,
            "due_date": "2025-06-30",
        },
    },
    {
        "email": "From: Marcus Lee <m.lee@biotechlabs.org>\nSubject: URGENT: dashboard down\n\nOur production dashboard has been returning 500 errors since this morning and our analysts are blocked. We need this resolved today. Ticket #4471 already open.",
        "expected": {
            "sender": "Marcus Lee",
            "category": "support_request",
            "summary": "Production dashboard is down with 500 errors; needs resolution today.",
            "action_required": True,
            "due_date": "none",
        },
    },
    {
        "email": "From: Priya Nair <priya@growthpartners.io>\nSubject: Re: enterprise pricing\n\nThanks for the deck. Leadership is interested but wants to see SOC 2 docs before we go further. No rush on timing.",
        "expected": {
            "sender": "Priya Nair",
            "category": "sales",
            "summary": "Interested in enterprise plan but needs SOC 2 docs before proceeding.",
            "action_required": True,
            "due_date": "none",
        },
    },
    {
        "email": "From: HR <people@acme.co>\nSubject: Office closed Monday\n\nReminder: the office will be closed Monday for the holiday. No action needed, just a heads up. Enjoy the long weekend!",
        "expected": {
            "sender": "HR",
            "category": "internal_update",
            "summary": "Office is closed Monday for the holiday; informational only.",
            "action_required": False,
            "due_date": "none",
        },
    },
    {
        "email": "From: Tom Briggs <tom@vendorworks.com>\nSubject: Contract renewal by Aug 1\n\nOur MSA is up for renewal. To keep continuity, please sign and return the renewal addendum no later than August 1, 2025. Reach out with redlines.",
        "expected": {
            "sender": "Tom Briggs",
            "category": "sales",
            "summary": "MSA renewal addendum must be signed and returned by August 1.",
            "action_required": True,
            "due_date": "2025-08-01",
        },
    },
    {
        "email": "From: alerts@statuspage.io\nSubject: All systems operational\n\nThis is your weekly status digest. All monitored services were operational over the past 7 days with 100% uptime. No incidents reported.",
        "expected": {
            "sender": "alerts@statuspage.io",
            "category": "internal_update",
            "summary": "Weekly status digest: all services operational, 100% uptime, no incidents.",
            "action_required": False,
            "due_date": "none",
        },
    },
    {
        "email": "From: Sofia Reyes <sofia@designco.studio>\nSubject: Need feedback on mockups\n\nDropped v2 of the mockups in the shared drive. Could you review and send comments before our sync on June 25? Want to finalize before the build phase.",
        "expected": {
            "sender": "Sofia Reyes",
            "category": "support_request",
            "summary": "Requests review of v2 mockups before the June 25 sync.",
            "action_required": True,
            "due_date": "2025-06-25",
        },
    },
]

In [ ]:
from datetime import datetime, timezone

import pandas as pd

# Flatten each example's expected extraction into top-level columns, so the dataset's
# output IS the extraction dict the evaluators compare against — not a nested wrapper.
rows = [{"email": e["email"], **e["expected"]} for e in EMAILS]
df = pd.DataFrame(rows)
OUTPUT_KEYS = ["sender", "category", "summary", "action_required", "due_date"]

dataset = await px_client.datasets.create_dataset(
    name=f"email-extraction-{datetime.now(timezone.utc):%Y%m%d-%H%M%S}",
    dataframe=df,
    input_keys=["email"],
    output_keys=OUTPUT_KEYS,
)
print(f"Uploaded {len(df)} examples to dataset '{dataset.name}'")

# Define the extraction task

The task is what we hold *almost* constant: the same schema, the same prompt, the same parsing — **only the model changes**. We use OpenAI's structured outputs so every model is forced to return the exact same shape, which keeps the comparison about extraction quality rather than formatting luck.

`make_task(model)` returns a task function bound to one model. The experiment calls it once per example; the `input` it receives is the dataset example's input dict (`{"email": ...}`).

In [ ]:
from typing import Literal

from openai import AsyncOpenAI
from pydantic import BaseModel

openai_client = AsyncOpenAI()


class EmailExtraction(BaseModel):
    sender: str
    category: Literal["meeting", "invoice", "support_request", "sales", "internal_update"]
    summary: str
    action_required: bool
    due_date: str  # ISO date (YYYY-MM-DD) or the literal string "none"


PROMPT = (
    "Extract structured fields from the email below. "
    "category must be one of: meeting, invoice, support_request, sales, internal_update. "
    "due_date must be an ISO date (YYYY-MM-DD) if the email states a concrete deadline, "
    'otherwise the literal string "none". '
    "action_required is true if the email asks the recipient to do something.\n\nEMAIL:\n{email}"
)


def make_task(model: str):
    async def task(input) -> dict:
        response = await openai_client.beta.chat.completions.parse(
            model=model,
            messages=[{"role": "user", "content": PROMPT.format(email=input["email"])}],
            response_format=EmailExtraction,
            temperature=0,  # deterministic — a fair comparison varies only the model
        )
        return response.choices[0].message.parsed.model_dump()

    return task

In [ ]:
# Sanity-check the task on a single example before running the full experiment.
await make_task("gpt-4o-mini")(dataset.examples[0]["input"])

# Choose metrics that measure what you care about

This is where benchmarks quietly lie. We score the **same** outputs two ways:

- **`jaro_winkler`** — string similarity over the JSON of the output vs. the expected output. It's cheap and forgiving — the kind of "looks about right" metric people reach for first — and it's the right tool for the free-text `summary`, which can be correct while worded differently.
- **`field_accuracy`** — the fraction of the **operational** fields (`sender`, `category`, `action_required`, `due_date`) that match *exactly* (case-insensitive). This is what downstream code depends on: a `due_date` that's "close" is still a wrong date. We deliberately leave `summary` out of this one — exact-matching a summary would punish good extractions for harmless rewording.

The two metrics measure genuinely different things, so they *can* rank the models differently: a model might write better summaries (higher `jaro_winkler`) while miscategorizing more emails (lower `field_accuracy`), or vice versa. When they disagree, the metric that should decide is the one tied to your downstream needs — here, `field_accuracy`. We make this concrete with a deterministic example below, then look at how the two real models actually land.

In [ ]:
import json

import jarowinkler

# The free-text summary is scored by jaro_winkler; field_accuracy judges only the
# operational fields downstream code actually depends on.
OPERATIONAL_FIELDS = ["sender", "category", "action_required", "due_date"]


def jaro_winkler(output, expected) -> float:
    """Forgiving string similarity over the serialized JSON (captures summary wording)."""
    return jarowinkler.jarowinkler_similarity(
        json.dumps(output, sort_keys=True),
        json.dumps(expected, sort_keys=True),
    )


def field_accuracy(output, expected) -> float:
    """Fraction of OPERATIONAL fields that match exactly (case-insensitive)."""
    matches = sum(
        1
        for k in OPERATIONAL_FIELDS
        if str(output.get(k)).strip().lower() == str(expected[k]).strip().lower()
    )
    return matches / len(OPERATIONAL_FIELDS)


EVALUATORS = [jaro_winkler, field_accuracy]

Before running the models, here's the disagreement made deterministic. Two candidate extractions for the same invoice email — one with the right operational fields but a reworded `summary`, one that looks almost identical but has the `due_date` off by a day:

In [ ]:
expected = {
    "sender": "billing@cloudhost.com",
    "category": "invoice",
    "action_required": True,
    "due_date": "2025-06-30",
    "summary": "CloudHost invoice #88231 for $4,200 is due June 30.",
}
# A: every operational field right, but the summary is fully reworded (harmless).
candidate_a = {
    **expected,
    "summary": "Please arrange payment for the recent cloud hosting charges before the close of the current billing period.",
}
# B: summary identical, but the due_date is off by a day.
candidate_b = {**expected, "due_date": "2025-07-01"}

for name, cand in [
    ("A (right fields, reworded summary)", candidate_a),
    ("B (identical summary, due_date off by a day)", candidate_b),
]:
    print(
        f"{name}: jaro_winkler={jaro_winkler(cand, expected):.3f}  "
        f"field_accuracy={field_accuracy(cand, expected):.3f}"
    )

# field_accuracy prefers A (all operational fields correct); jaro_winkler prefers B
# (it looks almost identical). But B's one-day date slip is exactly what breaks
# downstream code — same outputs, opposite rankings, and the strict metric is right.

# Run the same harness across models

Same dataset, same evaluators, same prompt — we change only the `model` argument. That's what makes this a fair comparison instead of an anecdote.

In [ ]:
experiment_4o = await px_client.experiments.run_experiment(
    dataset=dataset,
    task=make_task("gpt-4o"),
    evaluators=EVALUATORS,
    experiment_name="gpt-4o",
)

In [ ]:
experiment_4o_mini = await px_client.experiments.run_experiment(
    dataset=dataset,
    task=make_task("gpt-4o-mini"),
    evaluators=EVALUATORS,
    experiment_name="gpt-4o-mini",
)

# Compare fairly

Phoenix prints a per-experiment summary, and the UI lets you compare both runs example-by-example. Here we also roll the scores up per metric so the disagreement is explicit.

In [ ]:
from collections import defaultdict


def average_scores(experiment) -> dict:
    sums, counts = defaultdict(float), defaultdict(int)
    for run in experiment["evaluation_runs"]:
        result = run.result
        if result and result.get("score") is not None:
            sums[run.name] += result["score"]
            counts[run.name] += 1
    return {name: sums[name] / counts[name] for name in sums}


summary = pd.DataFrame(
    {"gpt-4o": average_scores(experiment_4o), "gpt-4o-mini": average_scores(experiment_4o_mini)}
)
print(summary)

Open the experiment comparison in Phoenix to see the per-example detail behind these averages. With only eight examples the two models may or may not separate cleanly on this run — that's exactly why you *look* rather than assume. The headline lesson holds regardless:

- If the two metrics **rank the models differently**, the public-leaderboard instinct ("just take the higher-scoring model") could have led you to the wrong choice — *which* model is "better" depends on the metric, and the metric that should win is the one tied to your downstream needs (here, `field_accuracy`).
- If they **agree**, you now have evidence grounded in *your* data and *your* metric — not a borrowed number from a benchmark that never saw your task.

Either way, you trust the result because you built the harness.

# Reading the results in Phoenix

Open the dataset's **Experiments** tab to see the runs side by side. Each experiment is a row; each evaluator becomes its own **score column** (so `field_accuracy` sits next to `jaro_winkler`), alongside operational columns — average **latency**, **cost**, and **error rate** — that matter for a real model choice but never show up on a public leaderboard.

![Comparing the gpt-4o and gpt-4o-mini experiments in Phoenix.](https://storage.googleapis.com/arize-phoenix-assets/assets/images/email-eval-harness-experiments.png)

Notice the two metrics can disagree on the headline question: here `gpt-4o` leads on `field_accuracy` while `gpt-4o-mini` edges ahead on `jaro_winkler`. Sort by the metric tied to your downstream needs (`field_accuracy`) rather than the forgiving one, and click any row to drop into the example-level view: the input email, the model's extraction, and each evaluator's score for that single example. That's where you *see why* one model wins — a `due_date` the model dropped, a sender it over-captured — instead of trusting an aggregate.

![The example-level compare view in Phoenix showing input, reference, and model output with per-example scores.](https://storage.googleapis.com/arize-phoenix-assets/assets/images/email-eval-harness-example-level.png)

# Where to go next

The harness is reusable — each step below holds the dataset and evaluators constant and changes one thing at a time, so every comparison stays fair:

- **Iterate the prompt.** Vary `PROMPT` instead of `model` to find the wording that extracts most reliably.
- **Add models and providers.** Drop another model name — or another provider's client — into `make_task` and rerun. The leaderboard ranking rarely survives contact with your task.
- **Grow the dataset.** Move from eight inline examples to a representative sample exported from your real traffic, so the scores become stable enough to trust (property 3).
- **Make it a standing benchmark.** Rerun the same harness on every model upgrade or prompt change to catch regressions before they reach production.

# Takeaway

A public benchmark tells you how a model does on *someone else's* task, with *someone else's* metric. That number rarely transfers to production.

The fix isn't a better leaderboard — it's a small, trustworthy harness of your own:

1. **Representative data** — a dataset built from your real inputs.
2. **A metric that measures what you care about** — and the discipline to notice when a convenient metric (string similarity) disagrees with the real one (field accuracy).
3. **Enough examples** to make the score stable.
4. **A fair, reproducible setup** — same dataset, same evaluators, same prompt; vary only the model.

A Phoenix experiment gives you all four. Once you have it, comparing models — or prompts, or providers — is just swapping one argument and reading a number you can actually defend.